# VQE: H2 Ground State

Variational Quantum Eigensolver for the H2 ground state using a hardware-efficient ansatz with COBYLA optimisation and exact Statevector evaluation.

In [ ]:
import numpy as np
import qiskit as qk
import scipy.optimize as opt

## H2 Hamiltonian

In [ ]:
H2_PAULIS = [
    (-0.81261, "II"),
    (0.17120, "IZ"),
    (-0.22279, "ZI"),
    (0.17120, "ZZ"),
    (0.04532, "XX"),
]

HAMILTONIAN = qk.quantum_info.SparsePauliOp.from_list(
    [(label, complex(coeff)) for coeff, label in H2_PAULIS]
)
EXACT_GS_ENERGY = -1.380398

print("Pauli decomposition:")
for coeff, label in H2_PAULIS:
    print(f"  {coeff:+.5f} · {label}")
print(f"\nExact ground-state energy: {EXACT_GS_ENERGY:.6f}")

## Ansatz and energy function

In [ ]:
N_LAYERS = 3
N_PARAMS = 4 * N_LAYERS

def ansatz_circuit(params):
    n_layers = len(params) // 4
    qc = qk.QuantumCircuit(2)
    for layer in range(n_layers):
        base = layer * 4
        qc.ry(params[base + 0], 0)
        qc.rz(params[base + 1], 0)
        qc.ry(params[base + 2], 1)
        qc.rz(params[base + 3], 1)
        qc.cx(0, 1)
    return qc

def energy(params):
    qc = ansatz_circuit(params)
    sv = qk.quantum_info.Statevector.from_instruction(qc)
    return float(np.real(sv.expectation_value(HAMILTONIAN)))

## COBYLA optimisation

In [ ]:
rng = np.random.default_rng(42)
init_params = rng.uniform(0, 2 * np.pi, size=N_PARAMS)
print(f"Initial energy: {energy(init_params):.6f}")

history = []
def callback(xk):
    history.append(energy(xk))

result = opt.minimize(
    energy, init_params, method="COBYLA",
    options={"maxiter": 200, "rhobeg": 0.5}, callback=cb,
)

print(f"\nOptimised energy: {result.fun:.6f}")
print(f"Error vs exact:   {abs(result.fun - EXACT_GS_ENERGY):.6f}")

## Energy convergence and final state

In [ ]:
step = max(1, len(history) // 10)
for i in range(0, len(history), step):
    print(f"  iter {i + 1:>3d}  energy = {history[i]:.6f}")
if (len(history) - 1) % step != 0:
    print(f"  iter {len(history):>3d}  energy = {history[-1]:.6f}")

sv = qk.quantum_info.Statevector.from_instruction(ansatz_circuit(result.x))
probs = sv.probabilities_dict()
print("\nFinal state probabilities:")
for bs in sorted(probs.keys()):
    if probs[bs] > 0.001:
        print(f"  |{bs}>  P = {probs[bs]:.6f}")
print("\nOptimised circuit:")
print(ansatz_circuit(result.x).draw(output="text"))